## 1. New Experiment

Keeping the same trained model on MNAR (reproducibility experiment). MNAR: Missing Not At Random, white pixels are two times more likely to be missing. 

### 1.1. Compute statistics on the missing data given by the authors

In [ ]:
import numpy as np

hmnist_miss = np.load('external/datasets/hmnist/x_miss_nan.npy')
hmnist_miss = hmnist_miss.reshape(784, 70_000, 10).transpose(1,2,0)
mask = np.isnan(hmnist_miss)
hmnist_miss[mask] = 0.0

### 1.2. Authors' HMNIST MNAR

In [1]:
import numpy as np
from utils import create_imputation_plot, compute_mask_stastics

import sys
sys.path.append('..')
from imputegap.recovery.manager import TimeSeries
from imputegap.algorithms.gp_vae import gp_vae

# initialize the time series object
ts = TimeSeries()

seq_length = 10
nbr_features = 784

# model folder
# model_folder = 'imputegap_assets/models/20251205_152733'
# model_folder = 'external/models/251113_reproduce_hmnist'
model_folder = '/home/kaeslin/Downloads/hmnist_trained_v2'
validation_idx = 600_000
nr_samples_test = 122
sample_idx = 61

hmnist_miss_val = np.load("../external/datasets/hmnist/x_miss_nan.npy")[:,validation_idx:validation_idx+seq_length*nr_samples_test]
hmnist_full_val_gt = np.load("../external/datasets/hmnist/x_full.npy")[:,validation_idx:validation_idx+seq_length*nr_samples_test]

# load and normalize the dataset
ts.import_matrix(hmnist_miss_val)
# ts.normalize(normalizer="z_score")
ground_truth = hmnist_full_val_gt.T.reshape(-1,seq_length,nbr_features)

try:
  ts_m_imputed, ts_m_imputed_no_gt, results = gp_vae(ts.data, 
                        "../imputegap/wrapper/AlgoPython/GPVAE/config_gpvae_hmnist.yaml", 
                        model_folder,
                        ground_truth=ground_truth,
                        return_no_gt_imputation=True,
                        verbose=False)

  print(results)
  print(compute_mask_stastics(np.isnan(hmnist_miss_val.T.reshape(-1, 10, 784))))
  
  create_imputation_plot((28,28,1),
                       10, 
                       ts.data.T.reshape(-1, seq_length, nbr_features),
                       ts_m_imputed_no_gt.T.reshape(-1, seq_length, nbr_features),
                       ts_m_imputed.T.reshape(-1, seq_length, nbr_features),
                       ground_truth,
                       sample_idx)  
finally:
  ts=None
  hmnist_miss_val = None
  hmnist_full_val_gt = None

2026-01-10 19:15:16.891952: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:479] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-01-10 19:15:17.048425: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:10575] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-01-10 19:15:17.049226: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1442] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-01-10 19:15:17.193831: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-01-10 19:15:19.491248: W tensorflow/compiler/tf

Checkpoint successfully restored.


Imputing progress:   0%|          | 0/2 [00:00<?, ?it/s]2026-01-10 19:15:29.147403: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:465] Loaded cuDNN version 8907
2026-01-10 19:15:29.463892: W external/local_tsl/tsl/framework/bfc_allocator.cc:296] Allocator (GPU_0_bfc) ran out of memory trying to allocate 1.10GiB with freed_by_count=0. The caller indicates that this is not a failure, but this may mean that there could be performance gains if more memory were available.
2026-01-10 19:15:29.463949: W external/local_tsl/tsl/framework/bfc_allocator.cc:296] Allocator (GPU_0_bfc) ran out of memory trying to allocate 1.34GiB with freed_by_count=0. The caller indicates that this is not a failure, but this may mean that there could be performance gains if more memory were available.
2026-01-10 19:15:29.992939: W external/local_tsl/tsl/framework/bfc_allocator.cc:296] Allocator (GPU_0_bfc) ran out of memory trying to allocate 4.32GiB with freed_by_count=0. The caller indicates that this

Model evaluation...


100%|██████████| 2/2 [00:02<00:00,  1.07s/it]


{'nll': 0.3402022155791351, 'mse': 0.1052862531276842}
((0.44792572766811645, 0.022531871365504545), (0.4479257276681164, 0.015794818474347896), 0.44792572766811645, (4.479257276681165, 1.7810154315420224))


### 1.3. Custom Contaminations

To contaminate effectively I need more data, seq_length of 10 is too small, already with the offset the start of the sequence is not contaminated. Solution, contaminate a larger set of data, and then select to visualize the imputation, in this way however, the nll and mse are not probably representative of the contamination. I think I could then save the contamination of the specific sample, and evaluate it individually.

Scattered

In [4]:
import numpy as np
from utils import create_imputation_plot, compute_mask_stastics

import sys
sys.path.append('..')
from imputegap.recovery.manager import TimeSeries
from imputegap.algorithms.gp_vae import gp_vae

# initialize the time series object
ts = TimeSeries()

seq_length = 10
nbr_features = 784

# model folder
# model_folder = 'imputegap_assets/models/20251205_152733'
model_folder = 'external/models/251113_reproduce_hmnist'

validation_idx = 600_000
nr_samples_test = 122
sample_idx = 61

rate_dataset = 1.0
rate_series = 0.4
offset=0.1

hmnist_full_val_gt = np.load("../external/datasets/hmnist/x_full.npy")[:,validation_idx:validation_idx+seq_length*nr_samples_test]

# load and normalize the dataset
ts.import_matrix(hmnist_full_val_gt)
# ts.normalize(normalizer="z_score")

print(ts.data.shape)
# Add missingness to the data, ts has shape (T, V)
ts_m = ts.Contamination.scattered(ts.data,rate_dataset=rate_dataset,
                                  rate_series=rate_series, 
                                  offset=offset)

ground_truth = hmnist_full_val_gt.T.reshape(-1,seq_length,nbr_features)

try:
  ts_m_imputed, ts_m_imputed_no_gt, results = gp_vae(ts_m[:, sample_idx*seq_length:sample_idx*seq_length+seq_length], 
                      "../imputegap/wrapper/AlgoPython/GPVAE/config_gpvae_hmnist.yaml", 
                      model_folder,
                      ground_truth=ground_truth[sample_idx:sample_idx+1],
                      return_no_gt_imputation=True,
                      verbose=False)

  print(results)
  print(compute_mask_stastics(np.isnan(ts_m[:, sample_idx*seq_length:sample_idx*seq_length+seq_length].T.reshape(-1, 10, 784))))

  create_imputation_plot((28,28,1),
                        10, 
                        ts_m[:, sample_idx*seq_length:sample_idx*seq_length+seq_length].T.reshape(-1, seq_length, nbr_features),
                        ts_m_imputed_no_gt.T.reshape(-1, seq_length, nbr_features),
                        ts_m_imputed.T.reshape(-1, seq_length, nbr_features),
                        ground_truth[sample_idx:sample_idx+1],
                        0)
  
finally:
  ts=None
  hmnist_miss_val = None
  hmnist_full_val_gt = None

(784, 1220)

(CONT) missigness pattern: SCATTER
	percentage of contaminated series: 100.0%
	rate of missing data per series: 40.0%
	security offset: [0-122]
	index impacted : 122 -> 610
Invalid Path to the model checkpoint!


Imputing progress: 100%|██████████| 1/1 [00:00<00:00,  7.84it/s]


Model evaluation...


100%|██████████| 1/1 [00:00<00:00,  8.11it/s]


{'nll': 0.6998331904092625, 'mse': 0.521308381821041}
((0.8110969387755101, 0.00175353661796781), (0.8110969387755103, 0.0), 0.8110969387755103, (8.110969387755102, 3.8447794022705795))


MCAR

In [7]:
import numpy as np
from utils import create_imputation_plot, compute_mask_stastics

import sys
sys.path.append('..')
from imputegap.recovery.manager import TimeSeries
from imputegap.algorithms.gp_vae import gp_vae

# initialize the time series object
ts = TimeSeries()

seq_length = 10
nbr_features = 784

# model folder
# model_folder = 'imputegap_assets/models/20251205_152733'
model_folder = 'external/models/251113_reproduce_hmnist'

validation_idx = 600_000
sample_idx = 61
nr_samples_test = 122

# contamination parameters
rate_dataset = 1.0
rate_series = 0.8
block_size = 3

hmnist_full_val_gt = np.load("../external/datasets/hmnist/x_full.npy")[:,validation_idx:validation_idx+seq_length*nr_samples_test]

# load and normalize the dataset
ts.import_matrix(hmnist_full_val_gt)
# ts.normalize(normalizer="z_score")

print(ts.data.shape)
# Add missingness to the data, ts has shape (T, V)
ts_m = ts.Contamination.mcar(ts.data,
                             rate_dataset=rate_dataset, 
                             rate_series=rate_series, block_size=block_size)

ground_truth = hmnist_full_val_gt.T.reshape(-1,seq_length,nbr_features)

try:
  ts_m_imputed, ts_m_imputed_no_gt, results = gp_vae(ts_m[:, sample_idx*seq_length:sample_idx*seq_length+seq_length], 
                      "../imputegap/wrapper/AlgoPython/GPVAE/config_gpvae_hmnist.yaml", 
                      model_folder,
                      ground_truth=ground_truth[sample_idx:sample_idx+1],
                      return_no_gt_imputation=True,
                      verbose=False)

  print(results)
  print(compute_mask_stastics(np.isnan(ts_m[:, sample_idx*seq_length:sample_idx*seq_length+seq_length].T.reshape(-1, 10, 784))))

  create_imputation_plot((28,28,1),
                        10, 
                        ts_m[:, sample_idx*seq_length:sample_idx*seq_length+seq_length].T.reshape(-1, seq_length, nbr_features),
                        ts_m_imputed_no_gt.T.reshape(-1, seq_length, nbr_features),
                        ts_m_imputed.T.reshape(-1, seq_length, nbr_features),
                        ground_truth[sample_idx:sample_idx+1],
                        0)
  
finally:
  ts=None
  hmnist_miss_val = None
  hmnist_full_val_gt = None

(784, 1220)

(CONT) missigness pattern: MCAR
	selected series: 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99, 100, 101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116, 117, 118, 119, 120, 121, 122, 123, 124, 125, 126, 127, 128, 129, 130, 131, 132, 133, 134, 135, 136, 137, 138, 139, 140, 141, 142, 143, 144, 145, 146, 147, 148, 149, 150, 151, 152, 153, 154, 155, 156, 157, 158, 159, 160, 161, 162, 163, 164, 165, 166, 167, 168, 169, 170, 171, 172, 173, 174, 175, 176, 177, 178, 179, 180, 181, 182, 183, 184, 185, 186, 187, 188, 189, 190, 191, 192, 193, 194, 195, 196, 197, 198, 199, 200, 201, 202, 203, 204, 205, 206, 207, 208, 209, 

Imputing progress: 100%|██████████| 1/1 [00:00<00:00,  8.23it/s]


Model evaluation...


100%|██████████| 1/1 [00:00<00:00,  8.76it/s]


{'nll': 0.6972572568551842, 'mse': 0.5039988574692945}
((0.8931122448979592, 0.007981881554674487), (0.8931122448979592, 0.0), 0.8931122448979592, (8.931122448979592, 1.9067149774977934))
